# 머신러닝 기반 텍스트 분류
1. 데이터 준비 : 로딩, 입력데이터, 출력데이터, 학습/테스트 데이터 분리, 특징추출
2. 학습-평가
3. 배포 준비

### 1. 데이터 준비

In [3]:
import pandas as pd
datafile = './data/Korean_movie_reviews_2016.csv'
data_df = pd.read_csv(datafile)
data_df.head()

,review,label
0,부산 행 때문 너무 기대하고 봤,0
1,한국 좀비 영화 어색하지 않게 만들어졌 놀랍,1
2,조금 전 보고 왔 지루하다 언제 끝나 이 생각 드,0
3,평 밥 끼 먹자 돈 니 내고 미친 놈 정신사 좀 알 싶어 그래 밥 먹다 먹던 숟가락...,1
4,점수 대가 과 이 엑소 팬 어중간 점수 줄리 없겠 클레멘타인 이후 최고 평점 조작 ...,0


In [4]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165384 entries, 0 to 165383
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  165384 non-null  object
 1   label   165384 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.5+ MB


In [5]:
# 입력 데이터, 정답 데이터 추출
review_list = list(data_df.review) # 입력 데이터
label_list = list(data_df.label) # 출력 데이터
len(review_list), len(label_list)

(165384, 165384)

In [6]:
# 학습 데이터, 테스트 데이터 분리
from sklearn.model_selection import train_test_split

train_X, test_X, train_y, test_y = train_test_split(review_list, label_list, test_size=0.1)
len(train_X), len(test_X), len(train_y), len(test_y)

(148845, 16539, 148845, 16539)

In [7]:
# 한국어 토크나이저 정의
from konlpy.tag import Okt
def korean_tokenizer(text):
    my_tags = ['Noun', 'Adjective', 'Verb']
    my_stopwords = []
    tokenizer = Okt().pos
    return [word for word, tag in tokenizer(text) if tag in my_tags and word not in my_stopwords]

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000)
vectorizer.fit(train_X)

TfidfVectorizer(max_features=1000)

In [9]:
len(vectorizer.get_feature_names_out()), vectorizer.get_feature_names_out()[:10]

(1000,
 array(['가', '가고', '가는', '가볍', '가서', '가슴', '가장', '가족', '가지', '각본'],
       dtype=object))

In [34]:
# 학습 데이터 특징 추출
train_X_fv = vectorizer.transform(train_X)

In [35]:
# 테스트 데이터 특징 추출
test_X_fv = vectorizer.transform(test_X)

In [37]:
print(train_X_fv)

  (np.int32(0), np.int32(82))	0.654691032329552
  (np.int32(0), np.int32(638))	0.32150881291337385
  (np.int32(0), np.int32(765))	0.6841138321992173
  (np.int32(1), np.int32(441))	0.7638617723453578
  (np.int32(1), np.int32(811))	0.6453798825106098
  (np.int32(2), np.int32(35))	0.28066346669946335
  (np.int32(2), np.int32(206))	0.664025299946901
  (np.int32(2), np.int32(365))	0.23910507369737297
  (np.int32(2), np.int32(390))	0.5453262089617977
  (np.int32(2), np.int32(882))	0.35460754228046937
  (np.int32(3), np.int32(327))	0.3755822485974645
  (np.int32(3), np.int32(420))	0.4229461823767201
  (np.int32(3), np.int32(455))	0.30592903649305125
  (np.int32(3), np.int32(664))	0.3998935273405469
  (np.int32(3), np.int32(681))	0.3352640313339492
  (np.int32(3), np.int32(692))	0.28581232538475926
  (np.int32(3), np.int32(791))	0.4279139894044199
  (np.int32(3), np.int32(927))	0.22213971818848235
  (np.int32(4), np.int32(639))	0.7683733452327545
  (np.int32(4), np.int32(878))	0.64000187682211

In [38]:
# 정답 데이터를 ndarray 변환 np.array(sequence)
import numpy as np
train_y = np.array(train_y)
test_y = np.array(test_y)
print(train_y[:10])

[0 0 0 0 0 0 0 0 0 1]


# 2. 머신러닝 - 모델 학습
1. 의사결정트리, Decision Tree (DT) 
2. 랜덤포레스트, RandomForest (RF)

In [14]:
# 모델별 정확도를 dataframe으로 저장하여 비교
import pandas as pd
score_df = pd.DataFrame(columns=['train', 'test'])
score_df

,train,test


In [15]:
def get_scores(model, train_X, train_y, test_X, test_y):
    train_score = model.score(train_X, train_y) * 100
    test_score = model.score(test_X, test_y) * 100
    return train_score, test_score

### 1. Decision Tree

In [41]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()
dtc.fit(train_X_fv, train_y)

DecisionTreeClassifier()

In [42]:
train_score, test_score = get_scores(dtc, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)

95.79428264301791 79.0374266884334


In [43]:
score_df.loc['DecisionTree'] = [train_score, test_score]
score_df

,train,test
DecisionTree,95.794283,79.037427
RandomForest,95.793611,83.529839
NavieBayes,85.319628,85.180482
LogisticRegression,86.263563,85.718605
SVM,86.223252,85.670234


### 2. Random Forest

In [39]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_jobs=-1)
rf.fit(train_X_fv, train_y)

RandomForestClassifier(n_jobs=-1)

In [40]:
train_score, test_score = get_scores(rf, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['RandomForest'] = [train_score, test_score]
score_df

95.79361080318452 83.52983856339561


,train,test
DecisionTree,98.112802,80.319245
RandomForest,95.793611,83.529839
NavieBayes,85.319628,85.180482
LogisticRegression,86.263563,85.718605
SVM,86.223252,85.670234


### 3. 나이브 베이즈 분류

In [21]:
from sklearn.naive_bayes import MultinomialNB

mnb = MultinomialNB()
mnb.fit(train_X_fv, train_y)


MultinomialNB()

In [22]:
train_score, test_score = get_scores(mnb, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['NavieBayes'] = [train_score, test_score]
score_df

85.3196278007323 85.18048249591874


,train,test
DecisionTree,98.112802,80.319245
RandomForest,98.112130,85.011186
NavieBayes,85.319628,85.180482


### 4. 로지스틱 회귀 분석, Logistic Regression

In [23]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(solver='liblinear')
lr.fit(train_X_fv, train_y)

LogisticRegression(solver='liblinear')

In [24]:
train_score, test_score = get_scores(lr, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['LogisticRegression'] = [train_score, test_score]
score_df

86.26356276663644 85.71860451055082


,train,test
DecisionTree,98.112802,80.319245
RandomForest,98.112130,85.011186
NavieBayes,85.319628,85.180482
LogisticRegression,86.263563,85.718605


### 5. 서포트 백터 머신, SVM

In [25]:
from sklearn.svm import LinearSVC
svm = LinearSVC()
svm.fit(train_X_fv, train_y)

LinearSVC()

In [26]:
train_score, test_score = get_scores(svm, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['SVM'] = [train_score, test_score]
score_df

86.22325237663341 85.67023399238164


,train,test
DecisionTree,98.112802,80.319245
RandomForest,98.112130,85.011186
NavieBayes,85.319628,85.180482
LogisticRegression,86.263563,85.718605
SVM,86.223252,85.670234


In [27]:
score_df.sort_values(by='test', ascending=False)

,train,test
LogisticRegression,86.263563,85.718605
SVM,86.223252,85.670234
NavieBayes,85.319628,85.180482
RandomForest,98.112130,85.011186
DecisionTree,98.112802,80.319245


### 6. 배포준비
- 기능 구현
- 모델 저장

In [ ]:
# review = '영화가 너무 재미있다'
review = '이게 영화냐? 나도 만들겠다'
# review = '영화 재미는 전혀 느껴지지 않는다'

def analyze_sentimant(review) :
    # 전처리 및 특징 벡터 추출

    

    review_fv = vectorizer.transform([review])
    print(review_fv)

    result = rf.predict(review_fv)
    print(result)

    show = '긍정' if result[0] >= 0.5 else '부정'
    return show
show = analyze_sentimant(review)    
print(f'{review} -> {show}')


  (np.int32(0), np.int32(128))	0.8209327543809424
  (np.int32(0), np.int32(626))	0.2621911022160921
  (np.int32(0), np.int32(686))	0.5072723516053582
[0]
이게 영화냐? 나도 만들겠다 -> 부정


In [30]:
reviews = [
    '영화가 너무 재미있다.',
    '이게 영화냐? 나도 만들겠다',
    '개꿀잼',
    '핵노잼'
]

for review in reviews:
    print(f'{review} -> {analyze_sentimant(review)}')

  (np.int32(0), np.int32(626))	0.28850957353384704
  (np.int32(0), np.int32(761))	0.9574770106792736
[1]
영화가 너무 재미있다. -> 긍정
  (np.int32(0), np.int32(128))	0.8209327543809424
  (np.int32(0), np.int32(626))	0.2621911022160921
  (np.int32(0), np.int32(686))	0.5072723516053582
[0]
이게 영화냐? 나도 만들겠다 -> 부정

[1]
개꿀잼 -> 긍정
  (np.int32(0), np.int32(167))	0.6251006175304933
  (np.int32(0), np.int32(972))	0.7805441806605158
[0]
핵노잼 -> 부정


In [32]:
import joblib

vectorizer_file = './model/se_movie_vectorizer.pkl'
joblib.dump(vectorizer, vectorizer_file)

model_file = './model/se_movie_model.pkl'
joblib.dump(rf, model_file)

['./model/se_movie_model.pkl']